# 01 - Prepare data

This notebook loads, preprocesses, and embeds the data for rds-log-analysis.

First, copy the mock and private config files, and fill in your own values

```
cp config_mock.example.toml config_mock.toml
cp config_private.example.toml config_private.toml
```

In [ ]:
import datetime
import uuid

import numpy as np
from datasets import load_dataset
from typing import Iterable, Callable
from tqdm import tqdm
from rds_chat_analysis.preprocessing_validation import validate_processed_log

In [ ]:
wildchat_dataset = load_dataset("allenai/WildChat")["train"]
print("Wildchat schema:")
for k, v in wildchat_dataset[0].items():
    print(f"{k}: {type(v)}")

# Load mock and private data

In [ ]:
np.random.seed(42)

# Split into 1000 mock logs, and use the rest for private logs
MAX_PRIVATE_LOGS = 1000
MAX_MOCK_LOGS = 100

all_indices = np.arange(len(wildchat_dataset))
np.random.shuffle(all_indices)

assert MAX_MOCK_LOGS <= len(
    all_indices
), f"Dataset is too small for {MAX_MOCK_LOGS} mock logs"

mock_logs = wildchat_dataset.select(all_indices[:MAX_MOCK_LOGS])
private_logs = wildchat_dataset.select(all_indices[MAX_MOCK_LOGS:])

if MAX_PRIVATE_LOGS is not None:
    private_logs = private_logs.select(range(MAX_PRIVATE_LOGS))

print(f"Mock logs: {len(mock_logs)}")
print(f"Private logs: {len(private_logs):,}")

## Preprocessing

This cell should be re-implemented for every dataset, the code below is an example for wildchat v1.

In [ ]:
def clean_json_for_postgres(obj):
    # IMPORTANT, postgres does not allow null-bytes in json strings
    if isinstance(obj, str):
        return obj.replace("\x00", "")
    if isinstance(obj, (datetime.date, datetime.datetime)):
        return obj.isoformat()
    if isinstance(obj, list):
        return [clean_json_for_postgres(v) for v in obj]
    if isinstance(obj, dict):
        return {k: clean_json_for_postgres(v) for k, v in obj.items()}
    return obj


def process_wildchat_log(log_data: dict, max_content_length: int = 4096) -> list[dict]:
    """
    Convert a row of wildchat data into a list of messages with metadata

    Expected output format:
    [
        {
            "id": "1234" # A message ID, should be unique across all messages
            "text": "Hello, how are you?", # The log content
            "metadata": {
                "role": "user",
                "log_id": "5678",
                "timestamp": "2025-01-01T00:00:00Z",
                ... # Any other metadata, JSON serializable
            }
        },
        ...
    ]

    Args:
        log_data (dict): A single row of wildchat data
        max_content_length (int, optional): Maximum length of the "text" field, depends on the embedding model.
            Defaults to 4096 (~= 1000 tokens).

    Returns:
        list[dict]: A list of messages with metadata
    """

    # Common metadata shared by all messages in the log
    log_id = log_data["conversation_id"]
    base_metadata = {
        "log_id": log_id,
        "model": log_data["model"],
        "timestamp": log_data["timestamp"],
        "total_turns": log_data["turn"],
        "language": log_data["language"],
    }

    result = []
    for message_idx, message in enumerate(log_data["conversation"]):
        content = message["content"]
        if len(content) > max_content_length:
            content = content[:max_content_length]

        message_metadata = {
            **base_metadata,
            "role": message["role"],
            "language": message.get("language", base_metadata["language"]),
            "message_idx": message_idx,
        }

        # deterministic + unique ID for each message
        doc_id = uuid.uuid5(uuid.NAMESPACE_URL, f"{log_id}_{message_idx}")

        result.append(
            {
                "id": str(doc_id),
                "text": content,
                "metadata": message_metadata,
            }
        )

    # Clean result so it can be inserted into Postgres
    result = clean_json_for_postgres(result)
    return result


def preprocess_dataset(
    dataset: Iterable[dict],
    preprocess_fn: Callable[[dict], list[dict]],
    validate: bool = False,
    size: int | None = None,
) -> list[list[dict]]:
    results = []

    for log_data in tqdm(dataset, desc="Processing logs", total=size):
        processed = preprocess_fn(log_data)

        if validate:
            validate_processed_log(processed)

        results.append(processed)

    return results

In [ ]:
# Validate the preprocessing function

preprocessed_log = process_wildchat_log(mock_logs[0])
is_valid = validate_processed_log(preprocessed_log)

print("Your preprocessing function produces the expected output format!")

In [ ]:
# Preprocess dataset
# Putting validation to False will speed this cell up, but might result in rows with invalid schema

validate_while_preprocessing = True

# size kwarg is optional, only required for Iterables with no __len__ (for example, loading directly from a DB)
mock_preprocessed = preprocess_dataset(
    mock_logs,
    preprocess_fn=process_wildchat_log,
    validate=validate_while_preprocessing,
    size=len(mock_logs),
)
private_preprocessed = preprocess_dataset(
    private_logs,
    preprocess_fn=process_wildchat_log,
    validate=validate_while_preprocessing,
    size=len(private_logs),
)

# Embedding

We embed and save data to a parquet file as staging step, instead of pushing directly to the db

In [ ]:
from rds_chat_analysis.utils import load_config, load_embedder_from_config

config = load_config("config_mock.toml")

# NOTE cache_dir is only needed while testing, it caches every embedding as individual file on disk.
cache_dir = "./embedding_cache"
# cache_dir = None
embedder = load_embedder_from_config(config, cache_dir=cache_dir)

In [ ]:
from rds_chat_analysis.vector_store_utils import embed_to_parquet
from rds_chat_analysis import NOTEBOOK_DIR

embedding_dir = NOTEBOOK_DIR / "v2" / "embeddings_1000"

In [ ]:
embed_to_parquet(
    logs=mock_preprocessed,
    embedder=embedder,
    output_dir=embedding_dir / "mock",
    embedder_batch_size=4,
    rows_per_parquet_file=250_000,
)

In [ ]:
embed_to_parquet(
    logs=private_preprocessed,
    embedder=embedder,
    output_dir=embedding_dir / "private",
    embedder_batch_size=4,
    rows_per_parquet_file=250_000,
)